# Motor Simulation

Simulating a motor with various control strategies.

Imports:

In [ ]:
try:
    from pydrake.all import *
    from jax import config
    config.update("jax_enable_x64", True)
    import jax.numpy as jnp
    import time
    %matplotlib ipympl
    import matplotlib.pyplot as plt
    from pid import PIDController
    from rk4 import rk4_step
    from custom_plots import MotorPlot
    from control_utilities import check_controllability


    print('Imported packages.')
except Exception as e:
    print('Importing packages failed:')
    print(e)
    raise e

## Motor Dynamics

$x=[\theta, \dot\theta]^T \implies \dot x=[\dot\theta, \ddot\theta]^T$

Assuming ideal motor model: 

$\tau = \alpha V = I\ddot\theta$, $|\tau|\leq \tau_{max}$

$u = V \implies \ddot\theta=\frac{\alpha u}{I}$

Set constants and initial state. Initialize controllers:

In [ ]:
LIVE = False
alpha = 1 # (Nm)/V
I = 1 # kg * m
# max torque constraint
# approximately 35kg servo
tau_max = 350 # Nm
x0 = jnp.array([1., 0.])
tf = 5 # s
dt = 1e-2 # s
N = int(tf/dt)
# step function
pos_setpoint = [] # goal
i = 0
while i <= tf:
    pos_setpoint.append((i // 0.5) % 2 * jnp.pi / 2)
    i += dt


In [ ]:
def dyn_step(x, u):
    """
    Computes the dynamics of the system each timestep.
    Input:
        x: state
        u: control input
    Output:
        xdot: change of state
    """
    _, theta_dot = x
    theta_ddot = jnp.clip(alpha * u, -tau_max, tau_max) / I
    return jnp.array([theta_dot, theta_ddot])

## PID Control

Plot the motors position, velocity, and acceleration when controlled with a cascade(2-stage) PID controller.

In [ ]:
x = x0
pid_plot = MotorPlot()
pos_pid = PIDController(24, 0, 0)
vel_pid = PIDController(75, 0, 0)
area_diff = 0
max_abs_u = 0
sat_count = 0
for k in range(N):
    pos_pid.setpoint = pos_setpoint[k]
    vel_pid.setpoint = pos_pid.calculate(x[0], dt)
    u = vel_pid.calculate(x[1], dt)
    max_abs_u = max(max_abs_u, abs(u))
    if abs(u) >= tau_max:
        sat_count += 1
    area_diff += abs((x[0] - pos_setpoint[k]) * dt)
    x = rk4_step(dyn_step, x, dt, u)
    pid_plot.update_plot(k * dt, x, [pos_pid.setpoint, vel_pid.setpoint], LIVE)
    if LIVE:
        time.sleep(dt)
if not LIVE:
    print(f'IAE = {area_diff}')
    print(f'max |u| = {max_abs_u}')
    print(f'saturation ratio = {sat_count / N:.1%}')
    pid_plot.show_plot(N * dt, area_diff)

## State-Space Control
System is assumed to be fully-observable.

$\dot x=Ax+Bu$

$y=Cx+Du=x$

$\dot x=[\dot \theta, \ddot \theta]^T=\begin{pmatrix}0&1 \\ 0&0\end{pmatrix}x + \begin{pmatrix}0\\\frac{\alpha}{I}\end{pmatrix} u$

$y=\begin{pmatrix}1&0\\0&1\end{pmatrix}x + \begin{pmatrix}0\\0\end{pmatrix}u$

In [ ]:
x = x0
A = jnp.array([[0, 1], [0, 0]])
B = jnp.array([[0], [alpha / I]])
Q = jnp.array([[1880, 0], [0, 1.125]])
R = jnp.array([0.0006])
# Q = jnp.array([[1400, 0], [0, 25]])
# R = jnp.array([0.08])


In [ ]:
controllable, rank_C = check_controllability(A, B)
print(f"rank(C): {rank_C}")
print(f"Controllable: {controllable}")

In [ ]:
lqr_plot = MotorPlot()
(K, S) = LinearQuadraticRegulator(A, B, Q, R)
print(f'K = {K}')
print(f'S = {S}')
area_diff = 0
max_abs_u = 0
sat_count = 0
for k in range(N):
    u = float((-K @ (x - jnp.array([pos_setpoint[k], 0])))[0])
    max_abs_u = max(max_abs_u, abs(u))
    if abs(u) >= tau_max:
        sat_count += 1
    area_diff += abs((x[0] - pos_setpoint[k]) * dt)
    x = rk4_step(dyn_step, x, dt, u)
    lqr_plot.update_plot(k * dt, x, [pos_setpoint[k]], LIVE)
    if LIVE:
        time.sleep(dt)
if not LIVE:
    print(f'IAE = {area_diff}')
    print(f'max |u| = {max_abs_u}')
    print(f'saturation ratio = {sat_count / N:.1%}')
    lqr_plot.show_plot(N * dt, area_diff)